<img src="https://www.plus2net.com/images/top2.jpg" alt="plus2net logo" width="400">

# ID Card Generation in PDF Using Image and Data Practice Notebook

**Main Tutorial Link:** [Generate ID card in PDF using Image and Data from a template](https://www.plus2net.com/python/pdf-id-card1.php)

---

## 1. Environment Setup & Download Resources

Install `reportlab` inside Google Colab and set up sample directories and photos.

In [ ]:
!pip install reportlab > /dev/null
!mkdir -p output photos

# Download sample photos for testing batch generation
!wget -q -O photos/1.png https://www.plus2net.com/images/top2.jpg
!wget -q -O photos/2.png https://www.plus2net.com/images/top2.jpg
!wget -q -O photos/3.png https://www.plus2net.com/images/top2.jpg
!wget -q -O photos/4.png https://www.plus2net.com/images/top2.jpg

print("ReportLab environment initialized and photo assets ready!")

## 2. Define Card Template (`tk_id_card_temp.py`)

Define `my_temp(c)` function to draw card header labels, layout lines, copyright info, and a 35-degree rotated watermark.

In [ ]:
from reportlab.lib.units import inch

def my_temp(c):
    c.translate(inch, inch)
    c.setFont("Helvetica", 14)
    c.setStrokeColorRGB(0.1, 0.8, 0.1)
    c.setFillColorRGB(0, 0, 1)
    
    # Watermark
    c.rotate(35)
    c.setFillColorCMYK(0, 0, 0, 0.08)
    c.setFont("Helvetica", 100)
    c.drawString(-1.1 * inch, -0.5 * inch, "SAMPLE")
    c.rotate(-35)
    
    c.setFillColorRGB(1, 0, 0)
    c.setFont("Helvetica", 25)
    c.drawRightString(1.7 * inch, 2.3 * inch, 'Identity Card')
    
    c.setFillColorRGB(0, 0, 0)
    c.setFont("Helvetica", 24)
    c.drawRightString(0.3 * inch, 1.7 * inch, 'ID:')
    c.drawRightString(0.3 * inch, 1.3 * inch, 'Name:')
    c.drawRightString(0.3 * inch, 0.9 * inch, 'Class:')
    c.drawRightString(0.3 * inch, 0.5 * inch, 'Gender:')
    c.drawRightString(4.0 * inch, -0.5 * inch, 'Signature')
    
    c.line(-1.1, -0.7 * inch, 5 * inch, -0.7 * inch)
    c.setFont("Helvetica", 8)
    c.setFillColorRGB(1, 0, 0)
    c.drawString(0, -0.9 * inch, "\u00A9 plus2net.com")
    
    return c

print("ID card template function initialized!")

## 3. Generate Single ID Card PDF

Create a single-page PDF ID card using custom canvas size `(400, 300)`.

In [ ]:
from pathlib import Path
from reportlab.pdfgen import canvas

output_dir = Path('output')
pdf_single = output_dir / 'single_id_card.pdf'

c = canvas.Canvas(str(pdf_single), pagesize=(400, 300))
c = my_temp(c)

my_image = 'photos/1.png'
my_id, my_name, my_class, my_gender = 18, 'Kalu', 'Four', 'Female'

c.drawImage(my_image, 2.2 * inch, 0.7 * inch, width=1.5*inch, height=1.5*inch)

c.setFillColorRGB(0, 0, 1)
c.setFont("Helvetica", 20)
c.drawString(0.5 * inch, 1.7 * inch, str(my_id))
c.drawString(0.5 * inch, 1.3 * inch, my_name)
c.drawString(0.5 * inch, 0.9 * inch, my_class)
c.drawString(0.5 * inch, 0.5 * inch, my_gender)

c.showPage()
c.save()
print(f"Single ID Card PDF created: {pdf_single}")

## 4. Batch ID Card Generation from List Data

Loop over student records to produce a multi-page PDF document containing four formatted student ID cards.

In [ ]:
pdf_batch = output_dir / 'batch_id_cards.pdf'
img_path = Path('photos')

my_students = [
    [1, 'Alex J', 'Four', 'Female', '1.png'],
    [2, 'Ramana K', 'Three', 'Male', '2.png'],
    [3, 'Jack', 'Five', 'Female', '3.png'],
    [4, 'Ronne', 'Six', 'Female', '4.png']
]

c = canvas.Canvas(str(pdf_batch), pagesize=(400, 300))

for ids in my_students:
    c = my_temp(c)
    my_image = str(img_path / ids[4])
    c.drawImage(my_image, 2.2 * inch, 0.7 * inch, width=1.5*inch, height=1.5*inch)
    c.setFillColorRGB(0, 0, 1)
    c.setFont("Helvetica", 20)
    c.drawString(0.5 * inch, 1.7 * inch, str(ids[0]))
    c.drawString(0.5 * inch, 1.3 * inch, ids[1])
    c.drawString(0.5 * inch, 0.9 * inch, ids[2])
    c.drawString(0.5 * inch, 0.5 * inch, ids[3])
    c.showPage()

c.save()
print(f"Batch ID Cards PDF created: {pdf_batch}")

## 5. Generate ID Cards from SQLite BLOB Image Records

Create an in-memory SQLite database with a BLOB image column, query the binary data, and use `io.BytesIO` with `ImageReader` to draw student photos.

In [ ]:
import sqlite3
import io
from reportlab.lib.utils import ImageReader

# Create SQLite database with BLOB image column
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()
cursor.execute('''CREATE TABLE student_b (
    id INTEGER PRIMARY KEY,
    name TEXT,
    class TEXT,
    gender TEXT,
    photo BLOB
)''')

# Read sample photo binary
with open('photos/1.png', 'rb') as f:
    img_blob = f.read()

cursor.executemany('INSERT INTO student_b VALUES (?, ?, ?, ?, ?)', [
    (1, 'Alex J', 'Four', 'Female', img_blob),
    (2, 'Ramana K', 'Three', 'Male', img_blob),
    (3, 'Jack', 'Five', 'Female', img_blob)
])
conn.commit()

# Fetch records
cursor.execute('SELECT * FROM student_b')
r_set = cursor.fetchall()

pdf_blob = output_dir / 'sqlite_blob_id_cards.pdf'
c = canvas.Canvas(str(pdf_blob), pagesize=(400, 300))

for ids in r_set:
    my_image = ImageReader(io.BytesIO(ids[4])) # Pass binary BLOB data
    c = my_temp(c)
    c.drawImage(my_image, 2.2 * inch, 0.7 * inch, width=1.5*inch, height=1.5*inch)
    c.setFillColorRGB(0, 0, 1)
    c.setFont("Helvetica", 20)
    c.drawString(0.5 * inch, 1.7 * inch, str(ids[0]))
    c.drawString(0.5 * inch, 1.3 * inch, ids[1])
    c.drawString(0.5 * inch, 0.9 * inch, ids[2])
    c.drawString(0.5 * inch, 0.5 * inch, ids[3])
    c.showPage()

c.save()
print(f"SQLite BLOB ID Cards PDF created: {pdf_blob}")

---
### Visit [www.plus2net.com](https://www.plus2net.com) for more Python ReportLab tutorials and downloadable source code.